## Загрузка данных

In [1]:
import sqlite3

In [2]:
import seaborn as sns

In [3]:
conn = sqlite3.connect("truedl.db")

In [4]:
data = conn.execute("""\
select
    tournament_year,
    team_category,
    count(*) as cnt,
    avg(ratio_to_top10) as ratio_avg
from truedl_data
group by tournament_year, team_category
""").fetchall()

In [5]:
data

[(2001, '101-250', 1, 0.6635111876075731),
 (2001, '11-25', 1, 0.8710843373493976),
 (2001, '251-500', 1, 0.4208261617900172),
 (2001, '26-50', 1, 0.7481927710843372),
 (2002, '101-250', 1, 0.6346613545816733),
 (2002, '11-25', 1, 0.9203187250996016),
 (2002, '501-1000', 1, 0.4213147410358566),
 (2002, '51-100', 1, 0.7504268639726807),
 (2003, '1001-2000', 1, 0.21161290322580645),
 (2003, '101-250', 2, 0.6185107158482666),
 (2003, '11-25', 1, 0.9539748953974896),
 (2003, '2001-3000', 1, 0.23225806451612904),
 (2003, '251-500', 2, 0.4879553070002993),
 (2003, '26-50', 1, 0.8267782426778243),
 (2003, '501-1000', 2, 0.45351366697539336),
 (2003, '51-100', 2, 0.7035182435776308),
 (2004, '101-250', 1, 0.7786666666666666),
 (2004, '11-25', 1, 0.898),
 (2004, '51-100', 1, 0.7702857142857144),
 (2005, '101-250', 1, 0.5883190883190883),
 (2005, '11-25', 1, 0.8425925925925926),
 (2005, '251-500', 1, 0.42592592592592593),
 (2005, '26-50', 1, 0.7486772486772487),
 (2005, '51-100', 1, 0.6952861952

In [6]:
cat_dict = {
    "11-25": "1. 11-25",
    "26-50": "2. 26-50",
    "51-100": "3. 51-100",
    "101-250": "4. 101-250",
    "251-500": "5. 251-500",
    "501-1000": "6. 501-1000",
    "1001-2000": "7. 1001-2000",
    "2001-3000": "8. 2001-3000",
    "3001-5000": "9. 3001-5000",
    "5001+": "A. 5001+",
}

data_for_plot = []
for x in data:
    data_point = {
        "year": int(x[0]),
        "category": cat_dict[x[1]],
        "ratio": x[3]
    }
    if data_point["year"] > 2003:
        data_for_plot.append(data_point)
data_for_plot = sorted(data_for_plot, key=lambda x: (x["year"], x["category"]))

In [7]:
import pandas as pd
df = pd.DataFrame(data_for_plot)

In [8]:
df

,year,category,ratio
0,2004,1. 11-25,0.898000
1,2004,3. 51-100,0.770286
2,2004,4. 101-250,0.778667
3,2005,1. 11-25,0.842593
4,2005,2. 26-50,0.748677
...,...,...,...
86,2022,4. 101-250,0.729764
87,2022,5. 251-500,0.611469
88,2022,6. 501-1000,0.504561
89,2022,7. 1001-2000,0.379881


In [9]:
import plotly.express as px

In [10]:
fig = px.line(
    df,
    x="year",
    y="ratio",
    color="category"
)
fig.write_html('ratios.html', auto_open=True)

In [16]:
fig.write_image("ratios.png", width=1920, height=1370, scale=2.7)

In [12]:
help(fig.write_image)

Help on method write_image in module plotly.basedatatypes:

write_image(*args, **kwargs) method of plotly.graph_objs._figure.Figure instance
    Convert a figure to a static image and write it to a file or writeable
    object
    
    Parameters
    ----------
    file: str or writeable
        A string representing a local file path or a writeable object
        (e.g. a pathlib.Path object or an open file descriptor)
    
    format: str or None
        The desired image format. One of
          - 'png'
          - 'jpg' or 'jpeg'
          - 'webp'
          - 'svg'
          - 'pdf'
          - 'eps' (Requires the poppler library to be installed)
    
        If not specified and `file` is a string then this will default to the
        file extension. If not specified and `file` is not a string then this
        will default to `plotly.io.config.default_format`
    
    width: int or None
        The width of the exported image in layout pixels. If the `scale`
        property is 1

## Вычисление коэффициентов для truedl_tournaments

In [18]:
from collections import defaultdict

dd = defaultdict(list)
for rec in data:
    year = int(rec[0])
    if year < 2017:
        continue
    dd[rec[1]].append(rec[3])

In [52]:
def print_sorted_by_key(dict_):
    for cat in sorted(dict_, key=lambda x: (-dict_[x], x.split("-")[0])):
        print(cat, dict_[cat])

In [53]:
import numpy as np
dd_conv = {}
for cat in dd:
    dd_conv[cat] = np.mean(dd[cat])
dd_conv["1-10"] = 1.0
print_sorted_by_key(dd_conv)

1-10 1.0
11-25 0.9417728479009996
26-50 0.8858754436559563
51-100 0.8187509574445472
101-250 0.718558220302021
251-500 0.6212182468314692
501-1000 0.50472796782245
1001-2000 0.3702272600863823
5001+ 0.34187132509424806
2001-3000 0.26717237368925595
3001-5000 0.1907290405100533


In [54]:
dd_conv.pop("5001+", None)  # чтобы не мешала

0.34187132509424806

In [55]:
def convert_relative_to_cat(cat):
    result = {}
    val = dd_conv[cat]
    for key in dd_conv:
        result[key] = round(dd_conv[key] / val, 2)
    return result

In [56]:
dd_501_1000 = convert_relative_to_cat("501-1000")
print_sorted_by_key(dd_501_1000)

1-10 1.98
11-25 1.87
26-50 1.76
51-100 1.62
101-250 1.42
251-500 1.23
501-1000 1.0
1001-2000 0.73
2001-3000 0.53
3001-5000 0.38


In [57]:
dd_251_500 = convert_relative_to_cat("251-500")
print_sorted_by_key(dd_251_500)

1-10 1.61
11-25 1.52
26-50 1.43
51-100 1.32
101-250 1.16
251-500 1.0
501-1000 0.81
1001-2000 0.6
2001-3000 0.43
3001-5000 0.31


In [75]:
dd_251_500

{'1001-2000': 0.6,
 '101-250': 1.16,
 '11-25': 1.52,
 '2001-3000': 0.43,
 '251-500': 1.0,
 '26-50': 1.43,
 '3001-5000': 0.31,
 '501-1000': 0.81,
 '51-100': 1.32,
 '1-10': 1.61}

In [67]:
dd_101_250 = convert_relative_to_cat("101-250")
print_sorted_by_key(dd_101_250)

1-10 1.39
11-25 1.31
26-50 1.23
51-100 1.14
101-250 1.0
251-500 0.86
501-1000 0.7
1001-2000 0.52
2001-3000 0.37
3001-5000 0.27


In [58]:
def expected_points(dd_dict, difficulty_level, questions=36):
    result = {}
    for key in dd_dict:
        coeff = dd_dict[key]
        q_share = 1 - (difficulty_level / 10)
        projected_questions = int(round(questions * q_share * coeff))
        if projected_questions > questions:
            projected_questions = questions
        result[key] = projected_questions
    print_sorted_by_key(result)

In [59]:
expected_points(dd_251_500, 4.0)

1-10 35
11-25 33
26-50 31
51-100 29
101-250 25
251-500 22
501-1000 17
1001-2000 13
2001-3000 9
3001-5000 7


In [60]:
expected_points(dd_251_500, 3.0)

1-10 36
11-25 36
26-50 36
51-100 33
101-250 29
251-500 25
501-1000 20
1001-2000 15
2001-3000 11
3001-5000 8


In [61]:
expected_points(dd_501_1000, 4.0)

1-10 36
11-25 36
26-50 36
51-100 35
101-250 31
251-500 27
501-1000 22
1001-2000 16
2001-3000 11
3001-5000 8


In [62]:
expected_points(dd_251_500, 5.0)

1-10 29
11-25 27
26-50 26
51-100 24
101-250 21
251-500 18
501-1000 15
1001-2000 11
2001-3000 8
3001-5000 6


In [66]:
expected_points(dd_251_500, 5.0)

1-10 29
11-25 27
26-50 26
51-100 24
101-250 21
251-500 18
501-1000 15
1001-2000 11
2001-3000 8
3001-5000 6


In [69]:
expected_points(dd_101_250, 4.0)

1-10 30
11-25 28
26-50 27
51-100 25
101-250 22
251-500 19
501-1000 15
1001-2000 11
2001-3000 8
3001-5000 6


In [70]:
expected_points(dd_101_250, 5.5)

1-10 23
11-25 21
26-50 20
51-100 18
101-250 16
251-500 14
501-1000 11
1001-2000 8
2001-3000 6
3001-5000 4


In [72]:
expected_points(dd_501_1000, 5.5)

1-10 32
11-25 30
26-50 29
51-100 26
101-250 23
251-500 20
501-1000 16
1001-2000 12
2001-3000 9
3001-5000 6


In [74]:
expected_points(dd_251_500, 4.1)

1-10 34
11-25 32
26-50 30
51-100 28
101-250 25
251-500 21
501-1000 17
1001-2000 13
2001-3000 9
3001-5000 7


## Работа с truedl_tournaments

In [76]:
conn = sqlite3.connect("truedl_tournaments.db")

In [77]:
cur = conn.cursor()

In [78]:
cur.execute("drop table truedl")
conn.commit()

OperationalError: no such table: truedl

In [79]:
cur.execute("""\
create table truedl as
select
    tournament_id,
    min(tournament_name) as tournament_name,
    min(tournament_type) as tournament_type,
    count(*) as cnt,
    round(avg(truedl),1) as truedl
from truedl_tournament_data
group by tournament_id;
""")

In [80]:
conn.commit()

In [81]:
top50 = cur.execute("""\
select * from truedl
order by truedl desc
limit 100;
""").fetchall()

In [82]:
top50

[(4943, 'Первая лига ЧТ', 'Обычный', 3, 10.0),
 (5282, 'Чемпионат Санкт-Петербурга. Высшая лига', '(R) Обычный', 12, 10.0),
 (5283, 'Чемпионат Санкт-Петербурга. Первая лига', '(R) Обычный', 44, 10.0),
 (6149, 'Чемпионат Санкт-Петербурга. Первая лига', '(R) Обычный', 36, 10.0),
 (8688, '30 under 30 (УКР)', 'Синхрон', 1, 10.0),
 (4583, 'Регадо - Б', '(R) Синхрон', 6, 9.6),
 (8219, 'Чемпионат Грузии-2022', 'Асинхрон', 10, 9.3),
 (8225, 'Karjalaisia Arvoituksia', 'Асинхрон', 6, 9.3),
 (4664, 'Часы Судного Дня: Полночь', '(R) Синхрон', 92, 9.2),
 (6638, 'Узнать за 60 секунд!', 'Асинхрон', 1, 9.2),
 (3915, 'Посох Аарона', '(R) Синхрон', 153, 9.0),
 (4147, 'Гран-при Бауманки. Синхрон', 'Общий зачёт', 57, 9.0),
 (4426, 'Посох Аарона', '(R) Синхрон', 126, 9.0),
 (4693, 'Посошок Арончика', 'Синхрон', 81, 8.9),
 (4840, 'Молодёжный чемпионат Нижегородской области', 'Обычный', 12, 8.9),
 (4082, 'Серия Гран-при. Общий зачёт', 'Общий зачёт', 418, 8.8),
 (4614, 'Серия Гран-при. Общий зачёт', 'Общий за

In [83]:
cur.execute("""\
select * from truedl
where tournament_name = "Conversation 16"
order by truedl desc
limit 50;
""").fetchall()

[(6290, 'Conversation 16', 'Асинхрон', 56, 7.7)]

In [84]:
ovsch = cur.execute("""\
select * from truedl
where tournament_name like "%ОВСЧ%"
order by truedl desc
limit 50;
""").fetchall()

In [85]:
ovsch

[(8607, 'ОВСЧ. 3 этап (асинхрон, на английском)', 'Асинхрон', 2, 8.2),
 (8557, 'ОВСЧ. 2 этап (асинхрон, на узбекском)', 'Асинхрон', 1, 7.8),
 (4366, 'ОВСЧ. 3 этап', '(R) Синхрон', 755, 7.3),
 (4279, 'ОВСЧ', 'Общий зачёт', 1047, 7.2),
 (8558, 'ОВСЧ. 2 этап (асинхрон, на английском)', 'Асинхрон', 6, 7.2),
 (4685, 'ОВСЧ', 'Общий зачёт', 1181, 7.0),
 (5518, 'ОВСЧ', 'Общий зачёт', 1261, 7.0),
 (3458, 'ОВСЧ. 4 тур', '(R) Синхрон', 748, 6.5),
 (3866, 'ОВСЧ. 1 этап', '(R) Синхрон', 661, 6.2),
 (3871, 'ОВСЧ. 6 этап', '(R) Синхрон', 749, 6.2),
 (6255, 'ОВСЧ', 'Общий зачёт', 1330, 6.2),
 (3868, 'ОВСЧ. 3 этап', '(R) Синхрон', 792, 6.0),
 (3451, 'ОВСЧ. 2 тур', '(R) Синхрон', 797, 5.9),
 (3459, 'ОВСЧ. 5 тур', '(R) Синхрон', 766, 5.9),
 (8608, 'ОВСЧ. 3 этап (асинхрон, на беларусском)', 'Асинхрон', 7, 5.9),
 (4985, 'ОВСЧ. 5 этап', '(R) Синхрон', 864, 5.8),
 (4369, 'ОВСЧ. 6 этап', '(R) Синхрон', 811, 5.7),
 (7696, 'ОВСЧ. 1 этап (синхрон)', '(R) Синхрон', 589, 5.6),
 (3870, 'ОВСЧ. 5 этап', '(R) Синхрон'

In [43]:
len(ovsch)

39

In [86]:
kubbes = cur.execute("""\
select * from truedl
where (tournament_name like "%Кубок Бесконечности%"
or tournament_name like "%Кубок бесконечности%")
order by truedl desc
limit 50;
""").fetchall()

In [88]:
sorted(kubbes, key=lambda x: x[0], reverse=True)

[(8446,
  'Кубок Бесконечности: четвёртый этап (синхрон)',
  '(R) Синхрон',
  126,
  6.3),
 (8443,
  'Кубок Бесконечности: третий этап. Онлайн и асинхрон',
  'Асинхрон',
  32,
  4.6),
 (8442, 'Кубок Бесконечности: третий этап (синхрон)', '(R) Синхрон', 150, 5.2),
 (8265,
  'Кубок Бесконечности: второй этап. Онлайн и асинхрон',
  'Асинхрон',
  40,
  6.1),
 (8264, 'Кубок Бесконечности: второй этап (синхрон)', '(R) Синхрон', 140, 6.2),
 (8261,
  'Кубок Бесконечности: первый этап. Онлайн и асинхрон',
  'Асинхрон',
  34,
  5.4),
 (8260, 'Кубок Бесконечности: первый этап (синхрон)', '(R) Синхрон', 219, 5.8),
 (7036, 'Онлайн: Кубок Бесконечности. VII этап: Финал', 'Асинхрон', 186, 4.8),
 (6930, 'Онлайн: Кубок Бесконечности. VI этап', 'Асинхрон', 204, 4.1),
 (6859, 'Онлайн: Кубок Бесконечности. V этап', 'Асинхрон', 154, 5.2),
 (6728, 'Онлайн: Кубок Бесконечности. IV этап', 'Асинхрон', 129, 4.5),
 (6727, 'Онлайн: Кубок Бесконечности. III этап', 'Асинхрон', 158, 4.0),
 (6707, 'Онлайн: Кубок Беск